# Block 01: A character-level tokenizer

**Paper anchor:** §5.1 (the paper uses byte-pair encoding; we start simpler and build BPE in block 14)
**Builds on:** block 00
**Time:** about 2 hours, in short sessions
**You will write:** `build_vocab`, `encode`, `decode`
**Done when:** the checkpoint passes and you have train/val id arrays

> **How this notebook works.** Claude teaches each concept in chat, one step at a time (type `next`).
> You write every line of solution code, in the cells marked **YOUR CODE**. Checkpoint cells test behavior only
> and never contain a solution (please don't open `checks/` until you pass). Stuck or wrong? Paste your code and
> the output into chat: we find the misconception together. Log every mistake in the table at the bottom.

In [2]:
# provided: repo paths (works from notebooks/ locally and from the repo root on Colab)
import sys, pathlib
ROOT = pathlib.Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
for p in (ROOT, ROOT / "src"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import numpy as np
np.set_printoptions(precision=4, suppress=True)
from checks import b01

In [ ]:
# provided: where the chain keeps its files (see data/README.md)
from minitransformer.paths import TEXT_PATH, TRAIN_IDS_PATH, VAL_IDS_PATH, ensure_text
ensure_text()  # notebook 00 already did this; it only downloads again if the file is missing

## Big picture
A model only ever sees integers. A tokenizer is a reversible map between text and lists of ids.
Character level: tiny vocabulary, long sequences, nothing unknown. The paper's BPE (§5.1): about 37,000 tokens,
shorter sequences. Block 14 explores that trade-off, including for Khmer.

## Before you code
Answer in the cell below, in your own words. Guessing is fine; we check in chat.

1. Roughly how many distinct characters do you expect in tiny Shakespeare?
2. Why sort the characters before numbering them?
3. What should `encode` do with a character it has never seen? Decide, and write down why.

*My answers:*

1. Roughly there are 65 distinct characters. 
2. The reason that we sort the characters is the model only sees id, so if id is not identical, it would generate gibberish language even if it was trained on perfect English. For example, without sort training 'h' id 1, but in generation id 1 'a', so it'd generate a instead of h. This is why we need order, training and generating id have to be the same. 
3. 

In [3]:
with open(TEXT_PATH, "r", encoding="utf-8") as f:
    text = f.read()
print(len(text), len(set(text)))

1115394 65


In [8]:
# YOUR CODE: replace each `raise NotImplementedError` with your implementation

def build_vocab(text):
    """Character vocabulary.

    text: str
    returns: (stoi, itos)
        stoi: dict char -> id, ids 0 .. V-1 assigned in SORTED character order
        itos: dict id -> char, the inverse of stoi
    """
    chars = sorted(set(text))
    stoi = {ch: i for i, ch in enumerate(chars)}
    itos = {i: ch for i, ch in enumerate(chars)}
    return stoi, itos

def encode(text, stoi):
    """str -> list of int ids, one id per character."""
    ids = [stoi[ch] for ch in text]
    return ids

def decode(ids, itos):
    """list of int ids -> str. decode(encode(s, stoi), itos) == s."""
    text = ''.join(itos[i] for i in ids)
    return text


In [10]:
print(encode("hello", {'h': 0, 'e': 1, 'l': 2, 'o': 3}))

[0, 1, 2, 2, 3]


In [9]:
b01.check_tokenizer(build_vocab, encode, decode)

Checkpoint 01: character tokenizer
  PASS  vocab size = number of unique characters (14)
  PASS  ids follow sorted character order, so they are reproducible
  PASS  itos is the inverse of stoi
  PASS  encode returns integers
  PASS  one id per character
  PASS  decode(encode(text)) == text  (round trip)
  PASS  the empty string encodes to an empty sequence
  PASS  round trip works on Khmer text too (Python strings are Unicode code points)
  All checks passed.



True

## Apply it

In [16]:
# YOUR CODE: load the text from TEXT_PATH, build the vocabulary, print V and the vocabulary itself,
# encode the whole text into a NumPy int array `ids`, and split it: first 90% -> train_ids, last 10% -> val_ids.
# Save both with np.save to TRAIN_IDS_PATH and VAL_IDS_PATH so later notebooks can load them.

stoi, itos = build_vocab(text)
V = len(stoi )
encoded_ids = np.array(encode(text, stoi), dtype=np.int64)
n = len(encoded_ids)
train_ids = encoded_ids[:int(n * 0.9)]
val_ids = encoded_ids[int(n * 0.9):] 
np.save(TRAIN_IDS_PATH, train_ids)
np.save(VAL_IDS_PATH, val_ids)


In [17]:
print("V =", V)
print(repr("".join(itos[i] for i in range(V))))

V = 65
"\n !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz"


**Question.** Why split by position (first 90% / last 10%) instead of sampling random characters for validation?

*My answer:* The reason being if we randomly sample characters for validation. For example the word 'The', after T we might randomly sample b then a and if our output is The, but our validation is randomly sampled to be 'tba'. We would get wrong while we were indeed correct.

In [18]:
s = "ROMEO: But soft, what light"
print(decode(encode(s, stoi), itos) == s)

True


In [20]:
print(stoi['a'], stoi['b'], stoi['z'])

39 40 64


## Explain it back
Without notes. Then paste your answers into chat.

1. What exactly does a tokenizer guarantee, and what doesn't it guarantee?
2. Trade-offs between character-level and subword tokens, in one sentence each.
3. Why would a Khmer tokenizer need different design choices from an English one? (We return to this in block 14.)

*My explanation:*

1. It guarantees that if we decode what we encode, we get the same string for every letter that is in the corpora. Problems occur when we face with character outside of our corpus: encode raises a keyerror. It also doesn't guarantee that the id numbers mean anything (a and b being neighbours is just alphabetical order), or that ids stay the same if the vocab is built from a different corpus. 
2. The Trade-offs between character-level and subword tokens is that subword token allows us to predict more for less token than character level. But the trade off is for if our vocab size is a million, we'd need to encode a million. but for character level we only need the set of characters it takes to write our entire corpus. 
3. 

## Mistakes log
Every wrong attempt is data. Copy finished rows into `journal/mistakes.md`.

| # | What I wrote | Symptom | The misconception | The fix |
|---|---|---|---|---|
| 1 | `encode("café", stoi)` | `KeyError: 'é'` | Assumed `encode` could take any text. The vocabulary only holds the 65 characters seen in the corpus, so any other character has no id. | Not a bug in the code: a closed character vocabulary cannot encode what it never saw. Block 14 (byte-level BPE) removes the problem. |

## Graduate
Copy `build_vocab`, `encode`, `decode` into `src/minitransformer/tokenizer.py`. Run `pytest -q tests/test_tokenizer.py`: `test_char_tokenizer` should flip from skipped to passed. Commit: `git commit -m "block 01: character tokenizer"`